# Anatomy of a Great Prompt

Decompose prompts into role, task, context, constraints, format, and examples — then compare weak vs strong instructions locally and via LangChain.


## 1. Overview

This guide covers:

- Six prompt components: role, task, context, constraints, output format, examples
- A reusable checklist for rewriting vague instructions
- Local scoring of prompt completeness (no API)
- Side-by-side bad vs good prompts on the same task with LangChain
- Self-contained setup (no shared project helpers required)


## 2. Motivation

Vague prompts produce vague outputs. In production, "make it better" is not testable. Breaking a prompt into explicit parts lets you version, diff, and evaluate changes like any other interface contract.

Strong prompts answer: **who** is speaking, **what** to do, **what data** to use, **what rules** apply, and **what shape** the answer must take.


## 3. Concepts

### 3.1 Glossary

| Part | Purpose |
|------|---------|
| **Role** | Persona or expertise ("You are a senior technical editor") |
| **Task** | Imperative action ("Rewrite the paragraph for clarity") |
| **Context** | Background facts, documents, or audience |
| **Constraints** | Must/must-not rules (length, tone, banned topics) |
| **Output format** | JSON, bullets, table, markdown sections |
| **Examples** | Input/output pairs demonstrating the pattern (optional here) |

### 3.2 How it works

The model conditions on the entire prompt as token context. Clear structure reduces ambiguity: the model does not "see" your intent — only the tokens you provide. Explicit format strings increase the probability of parseable output.

### 3.3 When to use structured prompts

**Use for:** any production prompt, especially extraction, classification, and customer-facing copy.

**Trade-offs:** longer prompts cost more tokens. Start minimal; add constraints when evaluations fail.

**Skip heavy structure for:** exploratory brainstorming where format does not matter.


## 4. Architecture

```mermaid
flowchart TD
    role[Role persona] --> prompt[ChatPromptTemplate]
    task[Task instruction] --> prompt
    context[Context data] --> prompt
    constraints[Constraints] --> prompt
    format[Output format] --> prompt
    examples[Examples optional] --> prompt
    prompt --> llm[ChatOpenAI]
    llm --> output[Structured completion]
```

### Prompt layers

```text
┌─────────────────────────────────────┐
│ Role + Task          (always)       │
├─────────────────────────────────────┤
│ Context              (when needed)  │
├─────────────────────────────────────┤
│ Constraints + Format (for reliability) │
├─────────────────────────────────────┤
│ Examples             (few-shot)     │
└─────────────────────────────────────┘

Chain shape:  prompt | llm
```

This notebook is self-contained: setup, prompts, and chains all live in the cells below.


## 5. Local Python Examples


In [1]:
# Prompt checklist scorer — no API required
role = "You are a professional customer-success editor."
task = "Rewrite the email below to be polite, concise, and action-oriented."
context = (
    "Original email:\n"
    "Hey — your invoice is wrong again. Fix it ASAP or we cancel."
)
constraints = "- Max 120 words\n- No blame language\n- Include a clear next step"
output_format = "Return only the rewritten email body (no preamble)."

bad_prompt = "Fix this email."

good_prompt = f"""Role: {role}

Task: {task}

Context:
{context}

Constraints:
{constraints}

Output format:
{output_format}"""

parts = {
    "role": role,
    "task": task,
    "context": context,
    "constraints": constraints,
    "output_format": output_format,
    "examples": "",
}
completeness = {k: bool(v.strip()) for k, v in parts.items()}

print("=== bad (string) ===")
print(bad_prompt)
print("completeness:", {"task": True})  # vague task only
print()
print("=== good (structured) ===")
print(good_prompt)
print("completeness:", completeness)


=== bad (string) ===
Fix this email.
completeness: {'task': True}

=== good (structured) ===
Role: You are a professional customer-success editor.

Task: Rewrite the email below to be polite, concise, and action-oriented.

Context:
Original email:
Hey — your invoice is wrong again. Fix it ASAP or we cancel.

Constraints:
- Max 120 words
- No blame language
- Include a clear next step

Output format:
Return only the rewritten email body (no preamble).
completeness: {'role': True, 'task': True, 'context': True, 'constraints': True, 'output_format': True, 'examples': False}


### 5.1 Bad vs good prompt strings


In [2]:
# Compare vague vs structured prompts as plain strings
TOPIC = "on-call runbook for API latency spikes"

bad = f"Write something about {TOPIC}."

good = f"""Role: You are an SRE writing internal documentation.
Task: Draft an on-call runbook section for API latency spikes.
Context: Service stack is Python/FastAPI behind an ALB; metrics in Datadog.
Constraints:
- 5 numbered steps max
- Each step: action + owner role (on-call vs platform)
- Mention rollback before root-cause deep dive
Output format: Markdown with ## heading and numbered list only.
"""

print("BAD length:", len(bad), "chars")
print(bad)
print()
print("GOOD length:", len(good), "chars")
print(good)


BAD length: 61 chars
Write something about on-call runbook for API latency spikes.

GOOD length: 394 chars
Role: You are an SRE writing internal documentation.
Task: Draft an on-call runbook section for API latency spikes.
Context: Service stack is Python/FastAPI behind an ALB; metrics in Datadog.
Constraints:
- 5 numbered steps max
- Each step: action + owner role (on-call vs platform)
- Mention rollback before root-cause deep dive
Output format: Markdown with ## heading and numbered list only.



## 6. LangChain Examples

```python
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

load_dotenv(root / ".env")
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)
```


In [3]:
# Setup: load .env and create ChatOpenAI (standalone — no project helpers)
import os
from pathlib import Path

from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

root = next(
    p
    for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (p / ".env").is_file() or (p / "requirements.txt").is_file()
)
load_dotenv(root / ".env")

api_key = os.getenv("OPENAI_API_KEY", "")
if not api_key.strip() or "your_openai_api_key" in api_key.lower():
    raise SystemExit("Set OPENAI_API_KEY in .env before running the API cells.")

MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
llm = ChatOpenAI(model=MODEL, temperature=0.2)
print("Model ready:", llm.model_name)


Model ready: gpt-4o-mini


In [4]:
# API comparison: weak vs structured prompt on the same input
sample_email = "Hey — your invoice is wrong again. Fix it ASAP or we cancel."

weak_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are helpful."),
        ("human", "Improve this email: {email}"),
    ]
)

strong_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Follow the user instructions exactly."),
        (
            "human",
            "Role: You are a professional customer-success editor.\n\n"
            "Task: Rewrite the email below to be polite, concise, and action-oriented.\n\n"
            "Context:\nOriginal email:\n{email}\n\n"
            "Constraints:\n"
            "- Max 120 words\n"
            "- No blame language\n"
            "- Include a clear next step\n\n"
            "Output format:\nReturn only the rewritten email body (no preamble).",
        ),
    ]
)

weak_chain = weak_prompt | llm
strong_chain = strong_prompt | llm

print("=== WEAK PROMPT OUTPUT ===")
weak_out = weak_chain.invoke({"email": sample_email})
print(weak_out.content)

print("\n=== STRONG PROMPT OUTPUT ===")
strong_out = strong_chain.invoke({"email": sample_email})
print(strong_out.content)


=== WEAK PROMPT OUTPUT ===


Subject: Urgent: Invoice Correction Needed

Hi [Recipient's Name],

I hope this message finds you well. 

I wanted to bring to your attention that there seems to be an error in the latest invoice we received. Could you please review and correct it at your earliest convenience? 

If we are unable to resolve this promptly, we may have to consider other options. 

Thank you for your attention to this matter. I look forward to your swift response.

Best regards,  
[Your Name]  
[Your Position]  
[Your Company]  
[Your Contact Information]  

=== STRONG PROMPT OUTPUT ===


Subject: Invoice Correction Needed

Dear [Recipient's Name],

I hope this message finds you well. I noticed some discrepancies in the recent invoice and would appreciate your assistance in correcting it at your earliest convenience. 

To ensure we can proceed smoothly, could you please review the invoice and provide an updated version by [specific date]? If you need any additional information from my side, feel free to reach out.

Thank you for your attention to this matter. I look forward to your prompt response.

Best regards,  
[Your Name]  
[Your Position]  
[Your Contact Information]  


## 7. Implementation notes

1. **`load_dotenv` + `ChatOpenAI`** — Load the key from `.env`, then build the model in the notebook.
2. **`ChatPromptTemplate` vars** — Keep `{email}` (and other fields) so one chain serves many inputs.
3. **System vs user** — Put stable policy in `system`; put task + data in `human`.
4. **Output format** — Ask for "JSON only" or "no preamble" when downstream code parses the reply.
5. **Evaluation** — Compare weak vs strong on the same input with fixed temperature for fair A/B.
6. **Reuse the chain** — Build `prompt | llm` once; call `.invoke` / `.batch` per row.


## 8. Best practices

- Write the **task** as an imperative verb phrase ("Extract", "Classify", "Summarize").
- Put **untrusted user content** in delimited blocks (`---`, XML tags) separate from instructions.
- Specify **audience** in role or context ("for a VP", "for on-call engineers").
- Define **done**: length limits, required fields, banned phrases.
- Version prompts in git; diff the composed prompt string, not just outcomes.
- Add **examples** only when zero-shot fails (see one-shot / few-shot notebooks).
- Configure model via `.env` (`OPENAI_MODEL`, `OPENAI_TEMPERATURE`) so notebooks stay portable.


## 9. Common failure modes

| Symptom | Likely cause | Fix |
|---------|--------------|-----|
| Rambling essay | No length or format constraint | Add `Output format` and max words/items |
| Wrong tone | Role missing or vague | Set explicit persona and audience |
| Ignores provided data | Context buried or unclear | Label context; use delimiters |
| Unparseable JSON | Format not strict enough | Prefer `with_structured_output` |
| Inconsistent sections | Mixed instructions and data | Separate system policy from user payload |
| Prompt bloat | Every field filled "just in case" | Add sections only when evals fail |


## 10. Validation checklist

1. Run the checklist scorer; confirm `bad` lacks role, constraints, and format.
2. Run string comparison; confirm good prompt specifies role, task, context, constraints, format.
3. With a valid API key, run weak vs strong email rewrite; compare tone and length.
4. Verify strong output follows "no preamble" when that constraint is set.
5. Confirm `OPENAI_API_KEY` (and optional `OPENAI_MODEL`) load from `.env`.


## 11. Summary

- Great prompts explicitly specify **role, task, context, constraints, and format**.
- Structure makes prompts testable, diffable, and easier to improve.
- Use LangChain as `ChatPromptTemplate | llm` for weak vs strong A/B on the same input.
- This notebook stands alone — no shared `assets` imports required.

**Next:** `Zero_Shot_Prompting.ipynb` — instructions without exemplars.
